## 1. Giải nén và kiểm tra ảnh/XML

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

# 1. Kết nối Google Drive
drive.mount("/content/drive")

# 2. Đường dẫn dữ liệu gốc
ZIP_PATH = Path(
    "/content/drive/MyDrive/Face Mask Detection/archive.zip"
)

# 3. Thư mục giải nén trên Colab
DATA_DIR = Path("/content/face_mask_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 4. Kiểm tra ZIP và giải nén
assert ZIP_PATH.exists(), "Không tìm thấy archive.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Số tệp trong ZIP:", len(z.namelist()))
    print("Một số tệp đầu tiên:", z.namelist()[:10])
    z.extractall(DATA_DIR)

# 5. Kiểm tra số lượng ảnh và file XML
images = sorted(
    p for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
)

annotations = sorted(DATA_DIR.rglob("*.xml"))

print("\n===== KẾT QUẢ KIỂM TRA =====")
print("Tổng số ảnh:", len(images))
print("Tổng số file XML:", len(annotations))
print("Thư mục dữ liệu:", DATA_DIR)
print("Ảnh mẫu:", images[0] if images else "Không có")
print("XML mẫu:", annotations[0] if annotations else "Không có")

## 2. Thống kê phân bố nhãn và box

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import xml.etree.ElementTree as ET

DATA_DIR = Path("/content/face_mask_data")
IMG_DIR = DATA_DIR / "images"
XML_DIR = DATA_DIR / "annotations"

image_files = sorted(IMG_DIR.glob("*.png"))
xml_files = sorted(XML_DIR.glob("*.xml"))

label_count = Counter()
images_per_label = Counter()
box_per_image = []
errors = []
image_sizes = Counter()

image_map = {p.stem: p for p in image_files}
xml_map = {p.stem: p for p in xml_files}

# Kiểm tra ảnh và file XML có khớp tên không
missing_images = sorted(set(xml_map) - set(image_map))
missing_xml = sorted(set(image_map) - set(xml_map))

for xml_path in xml_files:
    try:
        root = ET.parse(xml_path).getroot()
        image_path = image_map.get(xml_path.stem)

        if image_path is None:
            continue

        with Image.open(image_path) as img:
            width, height = img.size

        image_sizes[(width, height)] += 1

        objects = root.findall("object")
        box_per_image.append(len(objects))
        labels_in_image = set()

        for obj in objects:
            label = obj.findtext("name")
            box = obj.find("bndbox")

            if not label or box is None:
                errors.append((xml_path.name, "Thiếu nhãn hoặc box"))
                continue

            label_count[label] += 1
            labels_in_image.add(label)

            try:
                xmin = float(box.findtext("xmin"))
                ymin = float(box.findtext("ymin"))
                xmax = float(box.findtext("xmax"))
                ymax = float(box.findtext("ymax"))

                if not (
                    0 <= xmin < xmax <= width
                    and 0 <= ymin < ymax <= height
                ):
                    errors.append(
                        (xml_path.name, "Bounding box không hợp lệ")
                    )
            except (TypeError, ValueError):
                errors.append(
                    (xml_path.name, "Tọa độ bị thiếu hoặc sai")
                )

        for label in labels_in_image:
            images_per_label[label] += 1

    except Exception as e:
        errors.append((xml_path.name, str(e)))

print("===== 1. KIỂM TRA FILE =====")
print("Tổng ảnh:", len(image_files))
print("Tổng XML:", len(xml_files))
print("XML thiếu ảnh:", len(missing_images))
print("Ảnh thiếu XML:", len(missing_xml))

print("\n===== 2. PHÂN BỐ NHÃN =====")
for label, count in label_count.most_common():
    print(f"{label}: {count} objects")

print("\n===== 3. SỐ ẢNH CHỨA TỪNG NHÃN =====")
for label, count in images_per_label.most_common():
    print(f"{label}: {count} images")

print("\n===== 4. BOUNDING BOX =====")
print("Tổng số đối tượng:", sum(label_count.values()))
print("Ảnh không có đối tượng:", box_per_image.count(0))
print("Ít nhất trong một ảnh:",
      min(box_per_image) if box_per_image else 0)
print("Nhiều nhất trong một ảnh:",
      max(box_per_image) if box_per_image else 0)

print("\n===== 5. KÍCH THƯỚC ẢNH =====")
print("Số kích thước khác nhau:", len(image_sizes))
for size, count in image_sizes.most_common(10):
    print(f"{size}: {count} ảnh")

print("\n===== 6. KIỂM TRA LỖI =====")
print("Tổng lỗi:", len(errors))
for error in errors[:20]:
    print(error)

## 3. Kiểm tra 12 bounding box lỗi

In [ ]:
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET

DATA = Path("/content/face_mask_data")

error_files = [
    110, 231, 251, 457, 5, 501,
    603, 616, 706, 787, 93
]

total_errors = 0

for number in error_files:
    name = f"maksssksksss{number}"

    xml_path = DATA / "annotations" / f"{name}.xml"
    img_path = DATA / "images" / f"{name}.png"

    root = ET.parse(xml_path).getroot()

    with Image.open(img_path) as img:
        W, H = img.size

    for i, obj in enumerate(root.findall("object")):
        box = obj.find("bndbox")

        x1 = float(box.findtext("xmin"))
        y1 = float(box.findtext("ymin"))
        x2 = float(box.findtext("xmax"))
        y2 = float(box.findtext("ymax"))

        if not (0 <= x1 < x2 <= W and 0 <= y1 < y2 <= H):

            total_errors += 1

            print(f"\n[{total_errors}] {name}")
            print("Nhãn:", obj.findtext("name"))
            print("Kích thước ảnh:", W, H)
            print("Bounding box:", x1, y1, x2, y2)

            reasons = []
            if x1 < 0 or x2 > W:
                reasons.append("Vượt chiều rộng")
            if y1 < 0 or y2 > H:
                reasons.append("Vượt chiều cao")
            if x1 >= x2:
                reasons.append("Tọa độ X không hợp lệ")
            if y1 >= y2:
                reasons.append("Tọa độ Y không hợp lệ")

            print("Nguyên nhân:", ", ".join(reasons))

print("\nTỔNG SỐ BOX LỖI:", total_errors)

## 4. Sửa 12 bounding box, lưu XML sạch

In [ ]:
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET

DATA = Path("/content/face_mask_data")
CLEAN = Path("/content/face_mask_clean/annotations")
CLEAN.mkdir(parents=True, exist_ok=True)

fixed = 0
total_objects = 0

for xml_file in sorted((DATA / "annotations").glob("*.xml")):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    img_file = DATA / "images" / f"{xml_file.stem}.png"

    with Image.open(img_file) as img:
        W, H = img.size

    for obj in root.findall("object"):
        box = obj.find("bndbox")

        x1 = float(box.findtext("xmin"))
        y1 = float(box.findtext("ymin"))
        x2 = float(box.findtext("xmax"))
        y2 = float(box.findtext("ymax"))

        # Sửa đúng trường hợp vượt biên 1 pixel
        if x2 == W + 1 and 0 <= x1 < W and 0 <= y1 < y2 <= H:
            box.find("xmax").text = str(W)
            x2 = W
            fixed += 1

        # Không chấp nhận box lỗi còn sót lại
        assert 0 <= x1 < x2 <= W
        assert 0 <= y1 < y2 <= H

        total_objects += 1

    tree.write(
        CLEAN / xml_file.name,
        encoding="utf-8",
        xml_declaration=True
    )

# Kiểm tra
xml_count = len(list(CLEAN.glob("*.xml")))

assert fixed == 12
assert xml_count == 853
assert total_objects == 4072

print("===== KẾT QUẢ LÀM SẠCH =====")
print("Bounding box đã sửa:", fixed)
print("Tổng XML:", xml_count)
print("Tổng đối tượng:", total_objects)
print("Bounding box lỗi còn lại: 0")
print("Trạng thái: THÀNH CÔNG")

## 5. Cài công cụ chia tập có nhiều nhãn

In [ ]:
!pip -q install iterative-stratification

## 6. Chuyển XML sang YOLO và chia train/val/test

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import xml.etree.ElementTree as ET
import numpy as np
import shutil

# ===== 1. CẤU HÌNH =====

DATA_DIR = Path("/content/face_mask_data")
IMAGE_DIR = DATA_DIR / "images"
XML_DIR = Path("/content/face_mask_clean/annotations")

OUTPUT_DIR = Path("/content/face_mask_yolo")

CLASSES = [
    "with_mask",
    "without_mask",
    "mask_weared_incorrect"
]

CLASS_TO_ID = {
    name: index for index, name in enumerate(CLASSES)
}

SEED = 42

# ===== 2. ĐỌC DANH SÁCH ẢNH =====

image_files = sorted(IMAGE_DIR.glob("*.png"))

assert len(image_files) == 853, "Sai số lượng ảnh!"

# Ma trận nhãn: mỗi ảnh có thể chứa nhiều lớp
presence = np.zeros(
    (len(image_files), len(CLASSES)),
    dtype=int
)

for i, image_path in enumerate(image_files):

    xml_path = XML_DIR / f"{image_path.stem}.xml"
    assert xml_path.exists(), f"Thiếu {xml_path.name}"

    root = ET.parse(xml_path).getroot()

    for obj in root.findall("object"):
        label = obj.findtext("name")

        assert label in CLASS_TO_ID, f"Nhãn lạ: {label}"

        presence[i, CLASS_TO_ID[label]] = 1

# ===== 3. CHIA TRAIN 80% / TEMP 20% =====

indices = np.arange(len(image_files))

split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, temp_idx = next(
    split1.split(
        np.zeros((len(indices), 1)),
        presence
    )
)

# ===== 4. CHIA TEMP THÀNH VAL VÀ TEST =====

split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=SEED
)

val_part, test_part = next(
    split2.split(
        np.zeros((len(temp_idx), 1)),
        presence[temp_idx]
    )
)

splits = {
    "train": train_idx,
    "val": temp_idx[val_part],
    "test": temp_idx[test_part]
}

# Kiểm tra không có ảnh trùng giữa các tập
all_indices = np.concatenate(list(splits.values()))

assert len(all_indices) == 853
assert len(set(all_indices.tolist())) == 853

# ===== 5. CHUYỂN XML SANG YOLO =====

total_objects = 0
statistics = {}

for split_name, split_indices in splits.items():

    img_out = OUTPUT_DIR / "images" / split_name
    lbl_out = OUTPUT_DIR / "labels" / split_name

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    class_counts = Counter()

    for idx in split_indices:

        image_path = image_files[idx]
        xml_path = XML_DIR / f"{image_path.stem}.xml"

        with Image.open(image_path) as image:
            W, H = image.size

        root = ET.parse(xml_path).getroot()

        yolo_lines = []

        for obj in root.findall("object"):

            label = obj.findtext("name")
            class_id = CLASS_TO_ID[label]

            box = obj.find("bndbox")

            x1 = float(box.findtext("xmin"))
            y1 = float(box.findtext("ymin"))
            x2 = float(box.findtext("xmax"))
            y2 = float(box.findtext("ymax"))

            # Kiểm tra tọa độ đã sạch
            assert 0 <= x1 < x2 <= W
            assert 0 <= y1 < y2 <= H

            # Chuyển sang YOLO normalized coordinates
            x_center = ((x1 + x2) / 2) / W
            y_center = ((y1 + y2) / 2) / H
            box_width = (x2 - x1) / W
            box_height = (y2 - y1) / H

            assert 0 <= x_center <= 1
            assert 0 <= y_center <= 1
            assert 0 < box_width <= 1
            assert 0 < box_height <= 1

            line = (
                f"{class_id} "
                f"{x_center:.8f} "
                f"{y_center:.8f} "
                f"{box_width:.8f} "
                f"{box_height:.8f}"
            )

            yolo_lines.append(line)

            class_counts[label] += 1
            total_objects += 1

        # Lưu file nhãn YOLO
        label_path = lbl_out / f"{image_path.stem}.txt"

        label_path.write_text(
            "\n".join(yolo_lines) + "\n",
            encoding="utf-8"
        )

        # Sao chép ảnh
        shutil.copy2(image_path, img_out / image_path.name)

    statistics[split_name] = class_counts

# ===== 6. TẠO DATA.YAML =====

yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect
"""

(OUTPUT_DIR / "data.yaml").write_text(
    yaml_content,
    encoding="utf-8"
)

# ===== 7. THỐNG KÊ KẾT QUẢ =====

print("===== PHÂN CHIA DATASET =====")

for split_name, split_indices in splits.items():

    print(f"\n{split_name.upper()}")
    print("Số ảnh:", len(split_indices))

    for label in CLASSES:
        print(
            f"  {label}: "
            f"{statistics[split_name][label]} objects"
        )

print("\n===== TỔNG KẾT =====")

print("Tổng ảnh:", len(all_indices))
print("Tổng đối tượng:", total_objects)
print("Thư mục:", OUTPUT_DIR)

assert total_objects == 4072, "Mất đối tượng!"

for split_name, split_indices in splits.items():
    assert len(split_indices) > 0

    for label in CLASSES:
        assert statistics[split_name][label] > 0, (
            f"Tập {split_name} thiếu lớp {label}"
        )

print("\nTHÀNH CÔNG: Dataset YOLO đã sẵn sàng!")

## 7. Xem thử ảnh có bounding box

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt
from pathlib import Path

# ===== CẤU HÌNH =====
DATASET = Path("/content/face_mask_yolo")
SPLIT = "train"       # train, val hoặc test
NUM_IMAGES = 12       # Số ảnh muốn hiển thị
SEED = 42

CLASSES = [
    "with_mask",
    "without_mask",
    "mask_weared_incorrect"
]

# Màu BGR trong OpenCV
COLORS = [
    (0, 200, 0),      # Xanh lá
    (0, 0, 255),      # Đỏ
    (0, 215, 255)     # Vàng
]

IMAGE_DIR = DATASET / "images" / SPLIT
LABEL_DIR = DATASET / "labels" / SPLIT

images = sorted(IMAGE_DIR.glob("*.png"))
assert images, f"Không tìm thấy ảnh trong {IMAGE_DIR}"

random.seed(SEED)
selected = random.sample(images, min(NUM_IMAGES, len(images)))

cols = 3
rows = (len(selected) + cols - 1) // cols

fig, axes = plt.subplots(
    rows, cols, figsize=(18, rows * 4.5)
)

axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, image_path in zip(axes, selected):

    image = cv2.imread(str(image_path))
    H, W = image.shape[:2]

    label_path = LABEL_DIR / f"{image_path.stem}.txt"
    assert label_path.exists(), f"Thiếu nhãn: {label_path}"

    lines = label_path.read_text().strip().splitlines()

    for line in lines:
        values = line.split()
        class_id = int(values[0])
        xc, yc, bw, bh = map(float, values[1:5])

        # YOLO normalized -> Pixel coordinates
        x1 = int((xc - bw / 2) * W)
        y1 = int((yc - bh / 2) * H)
        x2 = int((xc + bw / 2) * W)
        y2 = int((yc + bh / 2) * H)

        color = COLORS[class_id]
        name = CLASSES[class_id]

        # Vẽ bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

        # Vẽ tên lớp
        font_scale = 0.4
        thickness = 1

        (tw, th), baseline = cv2.getTextSize(
            name,
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            thickness
        )

        text_y = max(y1, th + 5)

        cv2.rectangle(
            image,
            (x1, text_y - th - 5),
            (x1 + tw + 4, text_y + baseline),
            color,
            -1
        )

        cv2.putText(
            image,
            name,
            (x1 + 2, text_y - 3),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            thickness
        )

    # BGR -> RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax.imshow(image)
    ax.set_title(f"{image_path.name} | {len(lines)} objects")
    ax.axis("off")

# Ẩn các ô trống
for ax in axes[len(selected):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Cài Ultralytics và kiểm tra GPU

In [ ]:
!pip -q install ultralytics

import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATA_YAML = Path("/content/face_mask_yolo/data.yaml")

assert DATA_YAML.exists(), "Không tìm thấy data.yaml"
assert torch.cuda.is_available(), "Chưa bật GPU!"

print("Dataset sẵn sàng!")

## 9. Tùy chọn: khôi phục dataset sau khi Colab reset

> **Không bắt buộc:** Chỉ chạy cell này khi Colab reset khiến dataset ở `/content` biến mất. Cell sẽ tạo lại dữ liệu YOLO.

In [ ]:
!pip -q install iterative-stratification

from google.colab import drive
from pathlib import Path
from PIL import Image
from collections import Counter
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import xml.etree.ElementTree as ET
import numpy as np
import zipfile
import shutil

# ===== 1. KẾT NỐI GOOGLE DRIVE =====
drive.mount("/content/drive")

ZIP_PATH = Path(
    "/content/drive/MyDrive/Face Mask Detection/archive.zip"
)
RAW = Path("/content/face_mask_data")
OUT = Path("/content/face_mask_yolo")

assert ZIP_PATH.exists(), "Không tìm thấy archive.zip!"

# ===== 2. GIẢI NÉN =====
RAW.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW)

IMG_DIR = RAW / "images"
XML_DIR = RAW / "annotations"

images = sorted(IMG_DIR.glob("*.png"))

assert len(images) == 853
assert len(list(XML_DIR.glob("*.xml"))) == 853

print("Đã giải nén:", len(images), "ảnh")

# ===== 3. CẤU HÌNH NHÃN =====
CLASSES = [
    "with_mask",
    "without_mask",
    "mask_weared_incorrect"
]

class_map = {name: i for i, name in enumerate(CLASSES)}

# ===== 4. XÁC ĐỊNH NHÃN CỦA TỪNG ẢNH =====
presence = np.zeros((len(images), 3), dtype=int)

for i, img_path in enumerate(images):
    root = ET.parse(
        XML_DIR / f"{img_path.stem}.xml"
    ).getroot()

    for obj in root.findall("object"):
        label = obj.findtext("name")
        assert label in class_map
        presence[i, class_map[label]] = 1

# ===== 5. CHIA TRAIN / VAL / TEST =====
split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(np.zeros((len(images), 1)), presence)
)

split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=42
)

val_part, test_part = next(
    split2.split(
        np.zeros((len(temp_idx), 1)),
        presence[temp_idx]
    )
)

splits = {
    "train": train_idx,
    "val": temp_idx[val_part],
    "test": temp_idx[test_part]
}

all_ids = np.concatenate(list(splits.values()))
assert len(all_ids) == 853
assert len(set(all_ids.tolist())) == 853

# ===== 6. CHUYỂN XML -> YOLO =====
if OUT.exists():
    shutil.rmtree(OUT)

fixed = 0
total = 0
stats = {}

for split_name, indices in splits.items():

    img_out = OUT / "images" / split_name
    lbl_out = OUT / "labels" / split_name

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    counts = Counter()

    for idx in indices:
        img_path = images[idx]

        with Image.open(img_path) as img:
            W, H = img.size

        root = ET.parse(
            XML_DIR / f"{img_path.stem}.xml"
        ).getroot()

        lines = []

        for obj in root.findall("object"):
            label = obj.findtext("name")
            cid = class_map[label]

            box = obj.find("bndbox")

            x1 = float(box.findtext("xmin"))
            y1 = float(box.findtext("ymin"))
            x2 = float(box.findtext("xmax"))
            y2 = float(box.findtext("ymax"))

            # Sửa đúng lỗi xmax vượt 1 pixel
            if (
                x2 == W + 1
                and 0 <= x1 < W
                and 0 <= y1 < y2 <= H
            ):
                x2 = float(W)
                fixed += 1

            assert 0 <= x1 < x2 <= W
            assert 0 <= y1 < y2 <= H

            xc = (x1 + x2) / (2 * W)
            yc = (y1 + y2) / (2 * H)
            bw = (x2 - x1) / W
            bh = (y2 - y1) / H

            lines.append(
                f"{cid} {xc:.8f} {yc:.8f} "
                f"{bw:.8f} {bh:.8f}"
            )

            counts[label] += 1
            total += 1

        (lbl_out / f"{img_path.stem}.txt").write_text(
            "\n".join(lines) + "\n",
            encoding="utf-8"
        )

        shutil.copy2(img_path, img_out / img_path.name)

    stats[split_name] = counts

# ===== 7. TẠO DATA.YAML =====
yaml_text = f"""path: {OUT}
train: images/train
val: images/val
test: images/test

names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect
"""

(OUT / "data.yaml").write_text(
    yaml_text,
    encoding="utf-8"
)

# ===== 8. KIỂM TRA =====
assert fixed == 12
assert total == 4072
assert (OUT / "data.yaml").exists()

print("\n===== KẾT QUẢ KHÔI PHỤC =====")
print("Bounding box đã sửa:", fixed)

for name, indices in splits.items():
    print(f"\n{name.upper()}: {len(indices)} ảnh")
    for label in CLASSES:
        print(f"  {label}: {stats[name][label]}")

print("\nTổng ảnh:", len(images))
print("Tổng objects:", total)
print("data.yaml:", OUT / "data.yaml")
print("\nDATASET YOLO SẴN SÀNG!")

## 10. Huấn luyện YOLO11n

> Nếu bạn đã có `best.pt` và chỉ muốn convert, hãy **bỏ qua** cell train.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Chưa bật GPU!"
assert Path("/content/face_mask_yolo/data.yaml").exists()

SAVE_DIR = Path(
    "/content/drive/MyDrive/Face Mask Detection/runs"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/face_mask_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=20,
    seed=42,
    project=str(SAVE_DIR),
    name="baseline_yolo11n",
    exist_ok=True,
    save_period=10,
    plots=True
)

print("TRAIN HOÀN THÀNH!")
print("Model:", SAVE_DIR / "baseline_yolo11n/weights/best.pt")

## 11. Cài ONNX

In [ ]:
%pip install -q onnx onnxslim onnxruntime

## 12. Export best.pt sang best.onnx

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import onnx

# 1. Đường dẫn model đã train
PT_PATH = Path(
    "/content/drive/MyDrive/Face Mask Detection/"
    "runs/baseline_yolo11n/weights/best.pt"
)

assert PT_PATH.exists(), f"Không tìm thấy: {PT_PATH}"

# 2. Load model
model = YOLO(str(PT_PATH))

# 3. Convert PT sang ONNX
onnx_path = Path(model.export(
    format="onnx",
    imgsz=640,
    batch=1,
    dynamic=False,
    simplify=True
))

# 4. Kiểm tra file ONNX
assert onnx_path.exists(), "Export thất bại!"

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("\n===== CONVERT HOÀN THÀNH =====")
print("Model gốc:", PT_PATH)
print("Model ONNX:", onnx_path)
print("Dung lượng:", round(onnx_path.stat().st_size / 1024**2, 2), "MB")
print("ONNX hợp lệ: True")